# Notebook 03: Feature Engineering + NLP + Enrichment Externo

## Objetivo

Transformar las columnas brutas del dataset en features listas para modelado. 
Cada nueva columna tiene un **prefijo que indica su origen**:

| Prefijo | Origen | Ejemplo |
|---------|--------|---------|
| *(ninguno)* | Dato original de Raona / LinkedIn | Industry, Number of employees |
| `ai_` | Generado por la IA de enrichment de Raona | ai_SENIORITY, ai_FIT |
| `fe_` | Feature engineering propio (este notebook) | fe_company_age, fe_log_employees |
| `ext_` | Enrichment externo (este notebook) | ext_ms_maturity_score |
| `nlp_` | NLP sobre campos de texto (este notebook) | nlp_embedding_01, nlp_topic |
| `target_` | Variable objetivo | target_replied |

## Estructura del notebook

1. **Limpieza:** eliminar columnas vacias, corregir tipos
2. **Feature Engineering (fe_):** transformaciones de columnas existentes
3. **Enrichment externo (ext_):** scoring de stack Microsoft desde Technologies used
4. **NLP (nlp_):** embeddings, topics y keywords desde campos de texto
5. **Ensamblado final:** inventario completo y guardado

## Datos de entrada
- `modeling_dataset_raw.parquet` (10,946 filas, 47 columnas) -- de NB01

## Datos de salida
- `modeling_dataset_final.parquet` -- dataset listo para modelado en NB04

## Imports y Configuracion

In [1]:
import pandas as pd
import numpy as np
import os
import gc
import warnings
import re
warnings.filterwarnings('ignore')

# Rutas
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
WORKING_DATA = os.path.join(PROJECT_ROOT, '..', '_working', 'data')
CACHE_DIR = os.path.join(PROJECT_ROOT, '..', '_working', 'cache')
os.makedirs(CACHE_DIR, exist_ok=True)

# Cargar dataset
df = pd.read_parquet(os.path.join(WORKING_DATA, 'modeling_dataset_raw.parquet'))
print(f'Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'Positivos: {df["target_replied"].sum()} ({df["target_replied"].mean():.1%})')

Dataset cargado: 10,946 filas x 47 columnas
Positivos: 865 (7.9%)


---
## 3.1 Limpieza: eliminar columnas sin informacion

En NB02 identificamos columnas completamente vacias (100% nulos) o con cobertura insuficiente 
para ser utiles en el modelado. Las eliminamos para simplificar el dataset.

### Criterio de eliminacion:
- **100% nulos:** Gender, Revenue, Jobs posted from LinkedIn, ai_FIT_INFRA, ai_FIT_WORKPLACE, ai_FIT_MAITE
- **< 1% datos:** ai_FIT_IA (0.5%), ai_FIT_COLABORA (0.2%) -- ademas contienen texto largo, no puntuaciones
- **Columnas de nombre** (no son features): First name, Last name, Full name

Nota: **NO eliminamos** columnas con cobertura parcial como Technologies used (15%), 
ai_CONTACT_REPORT (43%) o Contact country (54%) -- estas aun pueden aportar valor predictivo.

In [2]:
# Columnas a eliminar
cols_to_drop = [
    # 100% nulos (sin informacion)
    'Gender', 'Revenue', 'Jobs posted from LinkedIn',
    # FIT por producto casi vacios (<1%) y son texto, no puntuaciones
    'ai_FIT_IA', 'ai_FIT_COLABORA', 'ai_FIT_INFRA', 'ai_FIT_WORKPLACE', 'ai_FIT_MAITE',
    # Nombres (no son features predictivas, se usan solo para identificacion)
    'First name', 'Last name', 'Full name',
]

# Verificar que existen antes de eliminar
existing_drops = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=existing_drops)

print(f'Columnas eliminadas: {len(existing_drops)}')
for c in existing_drops:
    print(f'  - {c}')
print(f'\nDataset tras limpieza: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Columnas eliminadas: 11
  - Gender
  - Revenue
  - Jobs posted from LinkedIn
  - ai_FIT_IA
  - ai_FIT_COLABORA
  - ai_FIT_INFRA
  - ai_FIT_WORKPLACE
  - ai_FIT_MAITE
  - First name
  - Last name
  - Full name

Dataset tras limpieza: 10,946 filas x 36 columnas


---
## 3.2 Feature Engineering (prefijo `fe_`)

Creamos nuevas features a partir de las columnas existentes. Cada transformacion 
tiene una justificacion de negocio o estadistica.

### Resumen de features a crear:

| Feature | Fuente | Logica | Justificacion |
|---------|--------|--------|---------------|
| `fe_seniority_ord` | ai_SENIORITY | Codificacion ordinal CLEVEL=4...JR=0 | Jerarquia tiene orden natural |
| `fe_contact_score_ord` | ai_Contact_score | High=3, Medium=2, Low=1, Disqualified=0 | Score tiene orden natural |
| `fe_company_score_ord` | ai_Company_score | Idem | Idem |
| `fe_fit_approved` | ai_FIT | 1 si APROBADO, 0 si no | Simplifica la variable categorica |
| `fe_fit_data_approved` | ai_FIT_DATA | 1 si SI, 0 si no | Idem para producto DATA |
| `fe_company_age` | Year founded | 2026 - Year founded | La antiguedad es mas intuitiva que el ano |
| `fe_log_employees` | Number of employees | log1p(valor) | Distribucion muy sesgada (3 a 199K) |
| `fe_company_size_bucket` | Number of employees | Categorias: micro/small/medium/large/enterprise | Segmenta por tamano de empresa |
| `fe_log_connections` | Number of connections | log1p(valor) | Distribucion sesgada, comprime cola |
| `fe_headcount_momentum` | Growths 6mo/1yr/2yr | Media ponderada (0.5/0.3/0.2) | Combina 3 senales de crecimiento |
| `fe_has_email` | Professional email | 1 si existe, 0 si no | Contactos con email profesional pueden ser mas accesibles |
| `fe_has_bio` | Profile bio | 1 si existe, 0 si no | Perfiles completos pueden indicar mayor actividad |
| `fe_microsoft_flag` | ai_Microsoft | 1 si usa Microsoft, 0 si no | Simplifica categorica a binaria |

### 3.2.1 Codificacion ordinal de ai_SENIORITY

Los niveles de seniority tienen un **orden jerarquico natural**: un C-Level tiene mas poder de 
decision que un Junior. Convertimos esta jerarquia en un numero para que el modelo pueda 
capturar esta relacion.

| Valor original | Codigo | Significado |
|---------------|--------|-------------|
| CLEVEL | 4 | C-Suite (CEO, CTO, CFO) |
| DIRECTOR | 3 | Director de area |
| MANAGER | 2 | Manager / responsable |
| LEAD | 1 | Team lead / senior individual |
| JR | 0 | Junior / individual contributor |

In [3]:
# fe_seniority_ord: codificacion ordinal
seniority_map = {'CLEVEL': 4, 'DIRECTOR': 3, 'MANAGER': 2, 'LEAD': 1, 'JR': 0}
df['fe_seniority_ord'] = df['ai_SENIORITY'].map(seniority_map)

print('=== fe_seniority_ord ===')
print(df['fe_seniority_ord'].value_counts().sort_index())
print(f'Nulos (SENIORITY no reconocido o ausente): {df["fe_seniority_ord"].isna().sum()}')

=== fe_seniority_ord ===
fe_seniority_ord
0.0    1869
1.0    1953
2.0    1473
3.0    2988
4.0    1251
Name: count, dtype: int64
Nulos (SENIORITY no reconocido o ausente): 1412


### 3.2.2 Codificacion ordinal de ai_Contact_score y ai_Company_score

Estas puntuaciones fueron generadas por la IA de Raona para evaluar la calidad del contacto 
y de la empresa respectivamente. Los valores son categoricos con un orden claro.

In [4]:
# fe_contact_score_ord
contact_score_map = {'High': 3, 'Medium': 2, 'Low': 1, 'Disqualified': 0}
df['fe_contact_score_ord'] = df['ai_Contact_score'].map(contact_score_map)

print('=== fe_contact_score_ord ===')
print(df['fe_contact_score_ord'].value_counts().sort_index())
unmapped = df['ai_Contact_score'].notna() & df['fe_contact_score_ord'].isna()
if unmapped.sum() > 0:
    print(f'Valores no mapeados: {df.loc[unmapped, "ai_Contact_score"].value_counts().to_dict()}')

=== fe_contact_score_ord ===
fe_contact_score_ord
0.0    4033
1.0    1266
2.0    1418
3.0    3147
Name: count, dtype: int64
Valores no mapeados: {'Missing Data': 1037}


In [5]:
# fe_company_score_ord
company_score_map = {'High': 3, 'Medium': 2, 'Low': 1, 'Disqualified': 0}
df['fe_company_score_ord'] = df['ai_Company_score'].map(company_score_map)

print('=== fe_company_score_ord ===')
print(df['fe_company_score_ord'].value_counts().sort_index())
unmapped = df['ai_Company_score'].notna() & df['fe_company_score_ord'].isna()
if unmapped.sum() > 0:
    print(f'Valores no mapeados: {df.loc[unmapped, "ai_Company_score"].value_counts().to_dict()}')

=== fe_company_score_ord ===
fe_company_score_ord
0.0     140
1.0    2237
2.0    2368
3.0    4679
Name: count, dtype: int64
Valores no mapeados: {'Missing Data': 540}


### 3.2.3 Simplificacion de ai_FIT y ai_FIT_DATA

El campo `ai_FIT` contiene valores como "APROBADO", "DESAPROBADO", "DUDA" (a veces con emojis). 
Lo simplificamos a una variable binaria: **aprobado o no**.

El campo `ai_FIT_DATA` indica si la empresa es un fit para la linea de producto DATA 
(Power BI, Fabric). Valores: SI, NO, COMPETITOR, DUDA.

In [6]:
# fe_fit_approved: 1 si APROBADO, 0 si no
def classify_fit(val):
    if pd.isna(val):
        return np.nan
    val_upper = str(val).upper()
    if 'APROBADO' in val_upper and 'DESAPROBADO' not in val_upper:
        return 1
    if val_upper.strip() == 'SI':
        return 1
    return 0

df['fe_fit_approved'] = df['ai_FIT'].apply(classify_fit)

print('=== fe_fit_approved ===')
print(df['fe_fit_approved'].value_counts())
print(f'Nulos: {df["fe_fit_approved"].isna().sum()}')

=== fe_fit_approved ===
fe_fit_approved
1.0    9246
0.0     943
Name: count, dtype: int64
Nulos: 757


In [7]:
# fe_fit_data_approved: 1 si la empresa es fit para DATA, 0 si no
fit_data_map = {'SI': 1, 'NO': 0, 'COMPETITOR': 0, 'DUDA': 0}
df['fe_fit_data_approved'] = df['ai_FIT_DATA'].map(fit_data_map)

print('=== fe_fit_data_approved ===')
print(df['fe_fit_data_approved'].value_counts())
print(f'Nulos: {df["fe_fit_data_approved"].isna().sum()}')

=== fe_fit_data_approved ===
fe_fit_data_approved
1.0    9796
0.0     392
Name: count, dtype: int64
Nulos: 758


### 3.2.4 fe_company_age (antiguedad de la empresa)

En lugar de usar el ano de fundacion directamente, calculamos la **antiguedad en anos**. 
Esto es mas interpretable: una empresa con 100 anos de historia tiene un perfil muy diferente 
a una startup de 2 anos.

Nota: en NB02 verificamos que los anos de fundacion antiguos (ej. 1156 para Hospital Santa Creu, 
1218 para Universidad de Salamanca) son **datos reales** de instituciones historicas espanolas. 
No los corregimos.

In [8]:
df['fe_company_age'] = 2026 - df['Year founded']

print('=== fe_company_age ===')
print(df['fe_company_age'].describe().round(1))
print(f'\nEmpresas mas antiguas:')
oldest = df.nlargest(5, 'fe_company_age')[['Company name', 'Year founded', 'fe_company_age']]
print(oldest.to_string(index=False))

=== fe_company_age ===
count    6996.0
mean       65.1
std        86.8
min         2.0
25%        29.0
50%        47.0
75%        66.0
max       870.0
Name: fe_company_age, dtype: float64

Empresas mas antiguas:
                                Company name  Year founded  fe_company_age
                         HOSPITAL SANTA CREU        1156.0           870.0
XARXA SANTA TECLA Sanitària, Social i Docent        1171.0           855.0
                    Universidad de Salamanca        1218.0           808.0
                    Universidad de Salamanca        1218.0           808.0
                      Barcelona City Council        1249.0           777.0


### 3.2.5 fe_log_employees (transformacion logaritmica)

El numero de empleados tiene una **distribucion muy sesgada a la derecha**: la mayoria de empresas 
tiene pocos empleados, pero algunas tienen mas de 100.000. La transformacion logaritmica `log1p(x)` 
comprime esta cola y hace que las diferencias relativas sean proporcionales.

Usamos `log1p` (log(1+x)) en lugar de `log` para evitar problemas con valores 0.

Ejemplo: log1p(10) = 2.4, log1p(100) = 4.6, log1p(1000) = 6.9, log1p(100000) = 11.5

In [9]:
df['fe_log_employees'] = np.log1p(df['Number of employees'])

print('=== fe_log_employees ===')
print(df['fe_log_employees'].describe().round(2))
print(f'Nulos: {df["fe_log_employees"].isna().sum()}')

=== fe_log_employees ===
count    9607.00
mean        6.61
std         1.37
min         1.39
25%         5.60
50%         6.45
75%         7.46
max        12.20
Name: fe_log_employees, dtype: float64
Nulos: 1339


### 3.2.6 fe_company_size_bucket (segmento por tamano)

Agrupamos las empresas en segmentos por numero de empleados. Esto permite al modelo 
capturar **diferencias no lineales** entre tamanos (ej. el comportamiento de una micropyme 
es cualitativamente diferente al de una enterprise, no solo cuantitativamente).

| Segmento | Empleados | Descripcion |
|----------|-----------|-------------|
| 0 - micro | < 10 | Micropyme |
| 1 - small | 10 - 50 | Pequena empresa |
| 2 - medium | 50 - 250 | Mediana empresa |
| 3 - large | 250 - 1000 | Gran empresa |
| 4 - enterprise | > 1000 | Corporacion |

In [10]:
bins = [0, 10, 50, 250, 1000, np.inf]
labels = [0, 1, 2, 3, 4]
df['fe_company_size_bucket'] = pd.cut(
    df['Number of employees'], bins=bins, labels=labels, right=False
).astype(float)  # float para permitir NaN

size_names = {0: 'micro (<10)', 1: 'small (10-50)', 2: 'medium (50-250)', 
              3: 'large (250-1K)', 4: 'enterprise (>1K)'}

print('=== fe_company_size_bucket ===')
for k, v in sorted(size_names.items()):
    count = (df['fe_company_size_bucket'] == k).sum()
    rate = df[df['fe_company_size_bucket'] == k]['target_replied'].mean() * 100
    print(f'  {v}: {count:,} contactos ({rate:.1f}% reply rate)')
print(f'  NaN: {df["fe_company_size_bucket"].isna().sum()}')

=== fe_company_size_bucket ===
  micro (<10): 2 contactos (0.0% reply rate)
  small (10-50): 136 contactos (6.6% reply rate)
  medium (50-250): 2,081 contactos (7.5% reply rate)
  large (250-1K): 3,653 contactos (8.0% reply rate)
  enterprise (>1K): 3,735 contactos (9.6% reply rate)
  NaN: 1339


### 3.2.7 fe_log_connections

Misma logica que para empleados: el numero de conexiones LinkedIn tiene una distribucion sesgada 
(mediana 545, max 30.000). Aplicamos log1p para normalizar.

In [11]:
df['fe_log_connections'] = np.log1p(df['Number of connections'])

print('=== fe_log_connections ===')
print(df['fe_log_connections'].describe().round(2))

=== fe_log_connections ===
count    10169.00
mean         5.77
std          2.14
min          0.00
25%          5.00
50%          6.30
75%          7.15
max         10.31
Name: fe_log_connections, dtype: float64


### 3.2.8 fe_headcount_momentum (momentum de crecimiento)

Combinamos tres senales de crecimiento de plantilla en un unico indicador, 
dando **mas peso al crecimiento reciente** (6 meses) que al historico (2 anos).

Formula: `0.5 * growth_6mo + 0.3 * growth_yearly + 0.2 * growth_2yr`

Empresas en crecimiento rapido pueden tener mas necesidades de herramientas Microsoft.

In [12]:
df['fe_headcount_momentum'] = (
    0.5 * df['Six months headcount growth'].fillna(0) +
    0.3 * df['Yearly headcount growth'].fillna(0) +
    0.2 * df['Two years headcount growth'].fillna(0)
)

# Marcar como NaN si las tres fuentes son nulas
all_null = (df['Six months headcount growth'].isna() & 
            df['Yearly headcount growth'].isna() & 
            df['Two years headcount growth'].isna())
df.loc[all_null, 'fe_headcount_momentum'] = np.nan

# Cap outliers al percentil 1/99
if df['fe_headcount_momentum'].notna().sum() > 100:
    p1 = df['fe_headcount_momentum'].quantile(0.01)
    p99 = df['fe_headcount_momentum'].quantile(0.99)
    before_cap = df['fe_headcount_momentum'].describe()
    df['fe_headcount_momentum'] = df['fe_headcount_momentum'].clip(lower=p1, upper=p99)
    print(f'Capping aplicado: [{p1:.1f}, {p99:.1f}]')

print('=== fe_headcount_momentum ===')
print(df['fe_headcount_momentum'].describe().round(2))

Capping aplicado: [-11.7, 33.9]
=== fe_headcount_momentum ===
count    9588.00
mean        5.30
std         6.27
min       -11.70
25%         1.50
50%         4.40
75%         8.30
max        33.90
Name: fe_headcount_momentum, dtype: float64


### 3.2.9 Features binarias

Creamos indicadores simples de presencia/ausencia de informacion. La existencia de ciertos datos 
puede ser una senal en si misma (ej. un contacto con email profesional puede indicar un perfil 
mas completo o accesible).

In [13]:
# fe_has_email: tiene email profesional
df['fe_has_email'] = df['Professional email'].notna().astype(int)

# fe_has_bio: tiene biografia en LinkedIn
df['fe_has_bio'] = df['Profile bio'].notna().astype(int)

# fe_microsoft_flag: 1 si usa Microsoft
def parse_microsoft(val):
    if pd.isna(val):
        return np.nan
    val_upper = str(val).upper()
    if val_upper in ['TRUE', 'YES', 'SI', '1']:
        return 1
    if val_upper in ['FALSE', 'NO', '0']:
        return 0
    # Puede contener texto explicativo
    if 'SI' in val_upper or 'YES' in val_upper or 'MICROSOFT' in val_upper:
        return 1
    return 0

df['fe_microsoft_flag'] = df['ai_Microsoft'].apply(parse_microsoft)

print('=== Features binarias ===')
for col in ['fe_has_email', 'fe_has_bio', 'fe_microsoft_flag']:
    ones = df[col].sum()
    total = df[col].notna().sum()
    print(f'  {col}: {ones:,} positivos de {total:,} ({ones/max(total,1)*100:.1f}%)')

=== Features binarias ===
  fe_has_email: 6,216 positivos de 10,946 (56.8%)
  fe_has_bio: 5,169 positivos de 10,946 (47.2%)
  fe_microsoft_flag: 0.0 positivos de 9,334 (0.0%)


### 3.2.10 Resumen de features fe_ creadas

In [14]:
fe_cols = [c for c in df.columns if c.startswith('fe_')]
print(f'Total features fe_ creadas: {len(fe_cols)}')
for col in fe_cols:
    nn = df[col].notna().sum()
    print(f'  {col}: {nn:,} no-nulos ({nn/len(df)*100:.1f}%) | tipo: {df[col].dtype}')

Total features fe_ creadas: 13
  fe_seniority_ord: 9,534 no-nulos (87.1%) | tipo: float64
  fe_contact_score_ord: 9,864 no-nulos (90.1%) | tipo: float64
  fe_company_score_ord: 9,424 no-nulos (86.1%) | tipo: float64
  fe_fit_approved: 10,189 no-nulos (93.1%) | tipo: float64
  fe_fit_data_approved: 10,188 no-nulos (93.1%) | tipo: float64
  fe_company_age: 6,996 no-nulos (63.9%) | tipo: float64
  fe_log_employees: 9,607 no-nulos (87.8%) | tipo: float64
  fe_company_size_bucket: 9,607 no-nulos (87.8%) | tipo: float64
  fe_log_connections: 10,169 no-nulos (92.9%) | tipo: float64
  fe_headcount_momentum: 9,588 no-nulos (87.6%) | tipo: float64
  fe_has_email: 10,946 no-nulos (100.0%) | tipo: int64
  fe_has_bio: 10,946 no-nulos (100.0%) | tipo: int64
  fe_microsoft_flag: 9,334 no-nulos (85.3%) | tipo: float64


---
## 3.3 Enrichment externo: Microsoft Stack Scoring (prefijo `ext_`)

La columna `Technologies used` contiene una lista de tecnologias que usa cada empresa 
(solo 15% de cobertura, pero para esos contactos es muy informativa).

Raona vende soluciones sobre el ecosistema Microsoft. Si una empresa ya usa tecnologias Microsoft, 
es mas probable que necesite los servicios de Raona. Si usa tecnologias competidoras (Google, AWS), 
es menos probable.

### Scoring de tecnologias:

| Tecnologia | Puntos | Razon |
|-----------|--------|-------|
| Azure | +3 | Pilar fundamental del stack Microsoft |
| Microsoft 365 / Office 365 | +2 | Suite de productividad base |
| Dynamics 365 | +3 | ERP/CRM Microsoft, alto valor |
| Power Platform / Power BI | +2 | Herramientas de datos Microsoft |
| SharePoint | +2 | Colaboracion y gestion documental |
| Teams | +1 | Comunicacion (muy comun) |
| Active Directory / Entra ID | +1 | Identidad Microsoft |
| Exchange | +1 | Email Microsoft |
| Google Workspace | -1 | Competidor directo |
| AWS | -1 | Competidor cloud |
| Salesforce | -1 | Competidor CRM |

In [15]:
# Diccionario de scoring de tecnologias
TECH_SCORES = {
    # Microsoft ecosystem (positivo)
    'azure': 3, 'microsoft azure': 3,
    'dynamics 365': 3, 'dynamics': 3,
    'microsoft 365': 2, 'office 365': 2, 'microsoft office': 2,
    'power bi': 2, 'power platform': 2, 'power apps': 2, 'power automate': 2,
    'sharepoint': 2,
    'teams': 1, 'microsoft teams': 1,
    'active directory': 1, 'entra id': 1, 'azure ad': 1,
    'exchange': 1, 'exchange online': 1,
    'windows server': 1, 'sql server': 1,
    'intune': 2, 'endpoint manager': 2,
    'copilot': 2,
    # Competidores (negativo)
    'google workspace': -1, 'google cloud': -1, 'gcp': -1,
    'aws': -1, 'amazon web services': -1,
    'salesforce': -1, 'hubspot': -1,
    'slack': -1, 'zoom': -1,
}

def score_tech_stack(tech_string):
    """Calcula el score de madurez Microsoft a partir de la lista de tecnologias."""
    if pd.isna(tech_string):
        return np.nan, np.nan
    
    tech_lower = str(tech_string).lower()
    ms_score = 0
    has_competitor = 0
    
    for tech, score in TECH_SCORES.items():
        if tech in tech_lower:
            if score > 0:
                ms_score += score
            else:
                has_competitor = 1
    
    return ms_score, has_competitor

# Aplicar scoring
tech_results = df['Technologies used'].apply(score_tech_stack)
df['ext_ms_maturity_score'] = tech_results.apply(lambda x: x[0])
df['ext_has_competitor_tech'] = tech_results.apply(lambda x: x[1])

print('=== ext_ms_maturity_score ===')
print(f'Cobertura: {df["ext_ms_maturity_score"].notna().sum():,} ({df["ext_ms_maturity_score"].notna().mean():.1%})')
print(df['ext_ms_maturity_score'].describe().round(1))

print(f'\n=== ext_has_competitor_tech ===')
print(df['ext_has_competitor_tech'].value_counts())

=== ext_ms_maturity_score ===
Cobertura: 1,642 (15.0%)
count    1642.0
mean        5.2
std         5.4
min         0.0
25%         0.0
50%         4.0
75%         8.0
max        22.0
Name: ext_ms_maturity_score, dtype: float64

=== ext_has_competitor_tech ===
ext_has_competitor_tech
1.0    1026
0.0     616
Name: count, dtype: int64


In [16]:
# Verificar: reply rate por nivel de madurez Microsoft
ms_data = df[df['ext_ms_maturity_score'].notna()].copy()
if len(ms_data) > 50:
    ms_data['ms_bucket'] = pd.cut(ms_data['ext_ms_maturity_score'], 
                                   bins=[-1, 0, 3, 6, 100], 
                                   labels=['0 (sin MS)', '1-3 (bajo)', '4-6 (medio)', '7+ (alto)'])
    grp = ms_data.groupby('ms_bucket')['target_replied'].agg(['count', 'mean']).round(3)
    grp.columns = ['contactos', 'reply_rate']
    print('=== Reply rate por madurez Microsoft ===')
    print(grp)

=== Reply rate por madurez Microsoft ===
             contactos  reply_rate
ms_bucket                         
0 (sin MS)         511       0.088
1-3 (bajo)         305       0.134
4-6 (medio)        240       0.062
7+ (alto)          586       0.109


---
## 3.4 NLP: Procesamiento de campos de texto (prefijo `nlp_`)

Los campos de texto generados por la IA de Raona contienen informacion rica que podemos 
convertir en features numericas para el modelo.

### Campos disponibles:

| Campo | Cobertura | Contenido |
|-------|-----------|----------|
| `ai_COMPANY_REPORT` | 91% | Informe sobre la empresa (sector, tamano, oportunidades) |
| `ai_CONTACT_REPORT` | 43% | Informe sobre el perfil del contacto |
| `ai_MOMENTUM` | 16% | Senales de timing (por que contactar ahora) |

### Pipeline NLP:

1. **Sentence Embeddings:** Convertimos el texto en vectores numericos de 384 dimensiones 
   usando el modelo `paraphrase-multilingual-MiniLM-L12-v2` (soporta espanol)
2. **UMAP:** Reducimos de 384 a 3 dimensiones para uso en el modelo
3. **BERTopic:** Agrupamos los textos en topics tematicos
4. **Keywords:** Extraemos senales especificas (urgencia, decision) por conteo de palabras clave

### Nota sobre circularidad

Estos textos fueron generados por IA, no escritos por los contactos. Usar NLP sobre texto 
generado por IA podria parecer circular. Lo justificamos porque:
- La IA de Raona sintetizo informacion de multiples fuentes (LinkedIn, CRM, web)
- Extraemos **patrones estructurales** (longitud, temas, urgencia), no repetimos la IA
- En NB04 compararemos el rendimiento del modelo con y sin features nlp_ para medir su contribucion incremental

### 3.4.1 Features basicas de texto (longitud, presencia)

In [17]:
# nlp_report_length: longitud del informe de empresa (proxy de visibilidad/importancia)
df['nlp_report_length'] = df['ai_COMPANY_REPORT'].str.len().fillna(0)

# nlp_contact_report_length: longitud del informe de contacto
df['nlp_contact_report_length'] = df['ai_CONTACT_REPORT'].str.len().fillna(0)

# nlp_has_momentum: tiene informacion de timing
df['nlp_has_momentum'] = df['ai_MOMENTUM'].notna().astype(int)

print('=== Features basicas de texto ===')
print(f'nlp_report_length: media={df["nlp_report_length"].mean():.0f}, '
      f'mediana={df["nlp_report_length"].median():.0f}')
print(f'nlp_contact_report_length: media={df["nlp_contact_report_length"].mean():.0f}')
print(f'nlp_has_momentum: {df["nlp_has_momentum"].sum():,} contactos con momentum')

=== Features basicas de texto ===
nlp_report_length: media=7383, mediana=7962
nlp_contact_report_length: media=1248
nlp_has_momentum: 1,755 contactos con momentum


### 3.4.2 nlp_urgency_score: senales de urgencia en MOMENTUM

El campo MOMENTUM contiene senales de por que contactar a una empresa AHORA. 
Buscamos palabras clave que indiquen urgencia o oportunidad de negocio.

In [18]:
# Palabras clave de urgencia/oportunidad
URGENCY_KEYWORDS = [
    'crecimiento', 'expansion', 'transformacion digital', 'transformación digital',
    'nuevo proyecto', 'inversion', 'inversión', 'presupuesto',
    'contratando', 'hiring', 'growth', 'expansion',
    'licitacion', 'licitación', 'concurso',
    'migracion', 'migración', 'modernizacion', 'modernización',
    'urgente', 'inmediato', 'prioridad',
    'nuevo cto', 'nuevo cio', 'nuevo director',
]

def count_urgency_keywords(text):
    if pd.isna(text):
        return 0
    text_lower = str(text).lower()
    return sum(1 for kw in URGENCY_KEYWORDS if kw in text_lower)

df['nlp_urgency_score'] = df['ai_MOMENTUM'].apply(count_urgency_keywords)

print('=== nlp_urgency_score ===')
print(df['nlp_urgency_score'].value_counts().sort_index().head(10))
print(f'\nContactos con urgency > 0: {(df["nlp_urgency_score"] > 0).sum()}')

=== nlp_urgency_score ===
nlp_urgency_score
0    10081
1      154
2      374
3      294
4       43
Name: count, dtype: int64

Contactos con urgency > 0: 865


### 3.4.3 Sentence Embeddings con paraphrase-multilingual-MiniLM-L12-v2

Convertimos `ai_COMPANY_REPORT` (91% cobertura) en vectores numericos de 384 dimensiones. 
Este modelo:
- Soporta **espanol** (y otros 50+ idiomas)
- Produce vectores de 384 dimensiones (mas eficiente que modelos de 768d)
- Captura el **significado semantico** del texto

Procesamos en batches de 500 para gestionar la memoria.

In [19]:
from sentence_transformers import SentenceTransformer

# Cargar modelo de embeddings multilingual
embeddings_cache = os.path.join(CACHE_DIR, 'company_report_embeddings.npy')

if os.path.exists(embeddings_cache):
    print('Cargando embeddings desde cache...')
    embeddings_full = np.load(embeddings_cache)
    print(f'Embeddings cargados: {embeddings_full.shape}')
else:
    print('Generando embeddings (puede tardar 2-5 minutos)...')
    model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    
    # Preparar textos: reemplazar NaN con cadena vacia
    texts = df['ai_COMPANY_REPORT'].fillna('').tolist()
    
    # Generar embeddings en batches
    BATCH_SIZE = 500
    all_embeddings = []
    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i+BATCH_SIZE]
        batch_emb = model.encode(batch, show_progress_bar=False)
        all_embeddings.append(batch_emb)
        print(f'  Batch {i//BATCH_SIZE + 1}/{(len(texts)-1)//BATCH_SIZE + 1} completado')
    
    embeddings_full = np.vstack(all_embeddings)
    
    # Guardar en cache
    np.save(embeddings_cache, embeddings_full)
    print(f'\nEmbeddings generados y guardados: {embeddings_full.shape}')
    
    # Liberar memoria
    del model
    gc.collect()

Generando embeddings (puede tardar 2-5 minutos)...


  Batch 1/22 completado


  Batch 2/22 completado


  Batch 3/22 completado


  Batch 4/22 completado


...

  Batch 18/22 completado


  Batch 19/22 completado


  Batch 20/22 completado


  Batch 21/22 completado


  Batch 22/22 completado

Embeddings generados y guardados: (10946, 384)


### 3.4.4 Reduccion de dimensionalidad con UMAP

384 dimensiones son demasiadas para un modelo tabular. Usamos **UMAP** (Uniform Manifold 
Approximation and Projection) para reducir a **3 dimensiones** que preservan las relaciones 
de vecindad del espacio original.

Por que UMAP y no PCA:
- UMAP preserva mejor la **estructura local** (clusters de empresas similares)
- PCA asume relaciones lineales; UMAP captura relaciones no lineales
- 3 dimensiones son suficientes para capturar las variaciones principales

In [20]:
import umap

umap_cache = os.path.join(CACHE_DIR, 'umap_3d.npy')

if os.path.exists(umap_cache):
    print('Cargando UMAP desde cache...')
    umap_3d = np.load(umap_cache)
else:
    print('Reduciendo dimensiones con UMAP (384 -> 3)...')
    reducer = umap.UMAP(n_components=3, random_state=42, n_neighbors=15, min_dist=0.1)
    umap_3d = reducer.fit_transform(embeddings_full)
    np.save(umap_cache, umap_3d)
    print('UMAP completado y guardado')

# Anadir al dataframe
df['nlp_embedding_01'] = umap_3d[:, 0]
df['nlp_embedding_02'] = umap_3d[:, 1]
df['nlp_embedding_03'] = umap_3d[:, 2]

# Para filas sin texto (COMPANY_REPORT nulo), los embeddings seran del texto vacio
# Marcamos esas filas para que el modelo sepa que no hay info real
no_text = df['ai_COMPANY_REPORT'].isna()
print(f'Filas sin texto (embeddings de cadena vacia): {no_text.sum()}')

print(f'\nnlp_embedding_01: mean={df["nlp_embedding_01"].mean():.2f}, std={df["nlp_embedding_01"].std():.2f}')
print(f'nlp_embedding_02: mean={df["nlp_embedding_02"].mean():.2f}, std={df["nlp_embedding_02"].std():.2f}')
print(f'nlp_embedding_03: mean={df["nlp_embedding_03"].mean():.2f}, std={df["nlp_embedding_03"].std():.2f}')

Reduciendo dimensiones con UMAP (384 -> 3)...


UMAP completado y guardado
Filas sin texto (embeddings de cadena vacia): 976

nlp_embedding_01: mean=3.82, std=7.18
nlp_embedding_02: mean=1.89, std=7.04
nlp_embedding_03: mean=5.32, std=6.57


### 3.4.5 Topic Modeling con BERTopic

BERTopic agrupa automaticamente los textos en **clusters tematicos** basandose en su contenido 
semantico. Cada contacto recibe un topic ID que representa el tema dominante de su empresa.

Ejemplo de topics que podrian emerger:
- Topic 0: Manufactura industrial
- Topic 1: Sector financiero
- Topic 2: Salud y hospitales
- etc.

In [21]:
from bertopic import BERTopic

topic_cache = os.path.join(CACHE_DIR, 'topic_assignments.npy')

if os.path.exists(topic_cache):
    print('Cargando topics desde cache...')
    topics = np.load(topic_cache)
else:
    print('Ejecutando BERTopic (puede tardar 2-5 minutos)...')
    
    # Preparar textos: solo los que tienen COMPANY_REPORT
    texts_for_topic = df['ai_COMPANY_REPORT'].fillna('sin informacion de empresa').tolist()
    
    # BERTopic con embeddings pre-calculados
    topic_model = BERTopic(
        nr_topics='auto',
        min_topic_size=30,
        language='multilingual',
        verbose=True,
    )
    
    topics, probs = topic_model.fit_transform(texts_for_topic, embeddings=embeddings_full)
    topics = np.array(topics)
    np.save(topic_cache, topics)
    
    # Mostrar topics encontrados
    print(f'\nTopics encontrados: {len(set(topics)) - (1 if -1 in topics else 0)}')
    print(f'Contactos sin topic (outliers): {(topics == -1).sum()}')
    
    # Top palabras por topic
    topic_info = topic_model.get_topic_info()
    print('\nTop 10 topics:')
    print(topic_info.head(11).to_string(index=False))
    
    del topic_model
    gc.collect()

df['nlp_topic'] = topics
print(f'\nnlp_topic: {df["nlp_topic"].nunique()} topics unicos')
print(df['nlp_topic'].value_counts().head(10))

2026-03-13 14:14:37,753 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


Ejecutando BERTopic (puede tardar 2-5 minutos)...


2026-03-13 14:14:42,913 - BERTopic - Dimensionality - Completed ✓


2026-03-13 14:14:42,913 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-03-13 14:14:43,106 - BERTopic - Cluster - Completed ✓


...

2026-03-13 14:14:45,686 - BERTopic - Topic reduction - Reducing number of topics


2026-03-13 14:14:45,702 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-03-13 14:14:48,331 - BERTopic - Representation - Completed ✓


2026-03-13 14:14:48,337 - BERTopic - Topic reduction - Reduced number of topics from 165 to 3



Topics encontrados: 2
Contactos sin topic (outliers): 1548

Top 10 topics:
 Topic  Count                         Name                                           Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

### 3.4.6 Resumen de features NLP

In [22]:
nlp_cols = [c for c in df.columns if c.startswith('nlp_')]
print(f'Total features nlp_ creadas: {len(nlp_cols)}')
for col in nlp_cols:
    nn = df[col].notna().sum()
    print(f'  {col}: {nn:,} no-nulos | tipo: {df[col].dtype}')

Total features nlp_ creadas: 8
  nlp_report_length: 10,946 no-nulos | tipo: float64
  nlp_contact_report_length: 10,946 no-nulos | tipo: float64
  nlp_has_momentum: 10,946 no-nulos | tipo: int64
  nlp_urgency_score: 10,946 no-nulos | tipo: int64
  nlp_embedding_01: 10,946 no-nulos | tipo: float32
  nlp_embedding_02: 10,946 no-nulos | tipo: float32
  nlp_embedding_03: 10,946 no-nulos | tipo: float32
  nlp_topic: 10,946 no-nulos | tipo: int64


---
## 3.5 Codificacion de ai_DEPARTMENT (Target Encoding)

El departamento es una variable categorica con varios niveles (IT, Operations, Other, etc.). 
Usamos **target encoding**: reemplazamos cada categoria por la tasa de respuesta media 
de ese departamento, con **suavizado** para evitar overfitting en categorias con pocas muestras.

Formula: `encoded = (count * mean_dept + global_mean * smoothing) / (count + smoothing)`

El parametro `smoothing` (usamos 20) controla cuanto se acerca al promedio global 
cuando hay pocas muestras.

In [23]:
# Target encoding con suavizado
SMOOTHING = 20
global_mean = df['target_replied'].mean()

dept_stats = df.groupby('ai_DEPARTMENT')['target_replied'].agg(['mean', 'count']).reset_index()
dept_stats.columns = ['ai_DEPARTMENT', 'dept_mean', 'dept_count']

# Suavizado bayesiano
dept_stats['fe_department_encoded'] = (
    (dept_stats['dept_count'] * dept_stats['dept_mean'] + SMOOTHING * global_mean) /
    (dept_stats['dept_count'] + SMOOTHING)
)

print('=== Target encoding de ai_DEPARTMENT ===')
print(f'Promedio global: {global_mean:.3f}')
print(dept_stats.sort_values('fe_department_encoded', ascending=False).to_string(index=False))

# Aplicar al dataframe
dept_map = dept_stats.set_index('ai_DEPARTMENT')['fe_department_encoded'].to_dict()
df['fe_department_encoded'] = df['ai_DEPARTMENT'].map(dept_map)

=== Target encoding de ai_DEPARTMENT ===
Promedio global: 0.079
ai_DEPARTMENT  dept_mean  dept_count  fe_department_encoded
  Sales & MKT   0.110681        1292               0.110199
           IT   0.089290        3763               0.089236
        Other   0.076997        2078               0.077016
   OPerations   0.076641        2179               0.076662
      Finance   0.066508         842               0.066799


---
## 3.6 Ensamblado del dataset final

Seleccionamos todas las columnas que pasaran al modelado, organizadas por origen.

In [24]:
# Inventario final de columnas
id_cols = ['LinkedIn profile ID', 'Company URN', 'Company name']
campaign_cols = ['Campaigns', 'Campaign engagement status']
raw_cols = [
    'Job title', 'Years in role', 'Years in company', 'Number of connections',
    'Contact country', 'Geo region',
    'Industry', 'Number of employees', 'Year founded',
    'Hiring on LinkedIn',
    'Six months headcount growth', 'Two years headcount growth', 'Yearly headcount growth',
]
ai_cols = [
    'ai_SENIORITY', 'ai_DEPARTMENT', 'ai_FIT', 'ai_FIT_DATA',
    'ai_Contact_score', 'ai_Company_score', 'ai_Microsoft',
]
text_cols = ['ai_CONTACT_REPORT', 'ai_COMPANY_REPORT', 'ai_MOMENTUM']
fe_cols = [c for c in df.columns if c.startswith('fe_')]
ext_cols = [c for c in df.columns if c.startswith('ext_')]
nlp_cols = [c for c in df.columns if c.startswith('nlp_')]
engagement_cols = ['Conversation tags']
target_cols = ['target_replied', 'target_replied_linkedin', 'target_replied_email']

# Columnas que NO pasan al dataset final (ya fueron transformadas)
drop_transformed = ['Profile bio', 'Professional email', 'Technologies used',
                     'Last LinkedIn post date', 'ai_FIT_clean']
drop_existing = [c for c in drop_transformed if c in df.columns]
df = df.drop(columns=drop_existing)

print('=== Inventario de columnas por origen ===')
print(f'  IDs:            {len(id_cols)}')
print(f'  Campaign:       {len(campaign_cols)}')
print(f'  Raw:            {len(raw_cols)}')
print(f'  AI-enriched:    {len(ai_cols)}')
print(f'  Text (NLP src): {len(text_cols)}')
print(f'  fe_ (eng.):     {len(fe_cols)}')
print(f'  ext_ (externo): {len(ext_cols)}')
print(f'  nlp_ (NLP):     {len(nlp_cols)}')
print(f'  Engagement:     {len(engagement_cols)}')
print(f'  Targets:        {len(target_cols)}')
total = len(id_cols) + len(campaign_cols) + len(raw_cols) + len(ai_cols) + len(text_cols) + \
        len(fe_cols) + len(ext_cols) + len(nlp_cols) + len(engagement_cols) + len(target_cols)
print(f'  TOTAL:          {total}')
print(f'  Actual df cols: {len(df.columns)}')

=== Inventario de columnas por origen ===
  IDs:            3
  Campaign:       2
  Raw:            13
  AI-enriched:    7
  Text (NLP src): 3
  fe_ (eng.):     14
  ext_ (externo): 2
  nlp_ (NLP):     8
  Engagement:     1
  Targets:        3
  TOTAL:          56
  Actual df cols: 56


In [25]:
# Tabla resumen detallada (para la memoria)
col_inventory = []
for col in df.columns:
    if col.startswith('fe_'):
        origin = 'Feature Engineering'
    elif col.startswith('ext_'):
        origin = 'Enrichment Externo'
    elif col.startswith('nlp_'):
        origin = 'NLP'
    elif col.startswith('ai_'):
        origin = 'AI-enriched (Raona)'
    elif col.startswith('target_'):
        origin = 'Target'
    elif col in id_cols:
        origin = 'ID'
    elif col in campaign_cols:
        origin = 'Campaign'
    else:
        origin = 'Raw (LinkedIn/HeyReach)'
    
    nn = df[col].notna().sum()
    col_inventory.append({
        'Columna': col,
        'Origen': origin,
        'Tipo': str(df[col].dtype),
        '% datos': f'{nn/len(df)*100:.0f}%',
    })

inv_df = pd.DataFrame(col_inventory)
print('=== Inventario completo de columnas ===')
print(inv_df.to_string(index=False))

print(f'\nResumen por origen:')
print(inv_df['Origen'].value_counts().to_string())

=== Inventario completo de columnas ===
                    Columna                  Origen    Tipo % datos
        LinkedIn profile ID                      ID  object    100%
                Company URN                      ID float64     89%
               Company name                      ID  object     91%
                  Campaigns                Campaign  object     89%
 Campaign engagement status                Campaign  object     89%
                  Job title Raw (LinkedIn/HeyReach)  object     93%
              Years in role Raw (LinkedIn/HeyReach) float64     82%
           Years in company Raw (LinkedIn/HeyReach) float64     78%
      Number of connections Raw (LinkedIn/HeyReach) float64     93%
            Contact country Raw (LinkedIn/HeyReach)  object     54%
                 Geo region Raw (LinkedIn/HeyReach)  object     93%
                   Industry Raw (LinkedIn/HeyReach)  object     89%
        Number of employees Raw (LinkedIn/HeyReach) float64     88%
        

---
## 3.7 Guardar dataset final

In [26]:
# Guardar
output_path = os.path.join(WORKING_DATA, 'modeling_dataset_final.parquet')
df.to_parquet(output_path, index=False)
file_size = os.path.getsize(output_path) / 1e6

print(f'Guardado: modeling_dataset_final.parquet')
print(f'  Tamano: {file_size:.1f} MB')
print(f'  Filas: {df.shape[0]:,}')
print(f'  Columnas: {df.shape[1]}')

Guardado: modeling_dataset_final.parquet
  Tamano: 35.0 MB
  Filas: 10,946
  Columnas: 56


---
## Resumen

### Que hemos hecho:

1. **Limpieza:** eliminamos 11 columnas vacias o inservibles
2. **Feature Engineering (fe_):** 14 nuevas features a partir de datos existentes
   - Codificaciones ordinales (seniority, contact score, company score)
   - Variables binarias (fit approved, has email, microsoft flag)
   - Transformaciones numericas (log employees, log connections, company age)
   - Combinaciones (headcount momentum)
   - Target encoding (department)
3. **Enrichment externo (ext_):** 2 features de scoring de stack tecnologico
   - Microsoft maturity score (basado en diccionario de tecnologias)
   - Flag de tecnologia competidora
4. **NLP (nlp_):** 7 features derivadas de campos de texto
   - 3 dimensiones de embeddings (UMAP de sentence-transformers)
   - Topic cluster (BERTopic)
   - Longitudes de texto y score de urgencia

### Total de features nuevas: ~23 (fe_ + ext_ + nlp_)

### Proximo paso: NB04 - Modelado (Lead Scoring, Segmentacion, Recomendacion)

In [27]:
# Resumen final
print('=' * 60)
print('RESUMEN NOTEBOOK 03')
print('=' * 60)
print(f'Dataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'\nFeatures por origen:')
for origin in ['Raw (LinkedIn/HeyReach)', 'AI-enriched (Raona)', 
               'Feature Engineering', 'Enrichment Externo', 'NLP',
               'Target', 'ID', 'Campaign']:
    count = (inv_df['Origen'] == origin).sum()
    if count > 0:
        print(f'  {origin}: {count}')

print(f'\nArchivos generados:')
for f in sorted(os.listdir(WORKING_DATA)):
    fpath = os.path.join(WORKING_DATA, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {f}: {size:.1f} MB')

print(f'\nArchivos en cache:')
for f in sorted(os.listdir(CACHE_DIR)):
    fpath = os.path.join(CACHE_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1e6
        print(f'  {f}: {size:.1f} MB')

RESUMEN NOTEBOOK 03
Dataset final: 10,946 filas x 56 columnas

Features por origen:
  Raw (LinkedIn/HeyReach): 14
  AI-enriched (Raona): 10
  Feature Engineering: 14
  Enrichment Externo: 2
  NLP: 8
  Target: 3
  ID: 3
  Campaign: 2

Archivos generados:


  conversation_analytics_ES.parquet: 0.3 MB
  daily_analytics_ES.parquet: 0.0 MB
  modeling_dataset_final.parquet: 35.0 MB
  modeling_dataset_raw.parquet: 37.5 MB
  replies_analytics_ES.parquet: 0.0 MB

Archivos en cache:
  company_report_embeddings.npy: 16.8 MB
  topic_assignments.npy: 0.1 MB
  umap_3d.npy: 0.1 MB
